# Workshop 2 — Data Profiling and Quality Risk Analysis
## Spotify Catalog + The Grammy Awards

Section 6.3 of the workshop brief: profiling is evidence gathering. It describes
the observed data and supports engineering decisions — it must not silently
alter the data. Every risk reported here is backed by a concrete number
computed below, not by inspection alone.

Sources profiled:
- `data/spotify_dataset.csv` — Spotify track catalog
- `data/the_grammy_awards.csv` — Grammy Awards nominations/winners, 1958-2019

Both files are read as-is; no cleaning happens in this notebook.


In [1]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

spotify = pd.read_csv("../data/spotify_dataset.csv")
grammy = pd.read_csv("../data/the_grammy_awards.csv")

print(f"Spotify: {spotify.shape[0]} rows, {spotify.shape[1]} columns")
print(f"Grammy:  {grammy.shape[0]} rows, {grammy.shape[1]} columns")


Spotify: 114000 rows, 21 columns
Grammy:  4810 rows, 10 columns


## 1. Structure

In [2]:
spotify.dtypes


Unnamed: 0            int64
track_id                str
artists                 str
album_name              str
track_name              str
popularity            int64
duration_ms           int64
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
track_genre             str
dtype: object

In [3]:
grammy.dtypes


year            int64
title             str
published_at      str
updated_at        str
category          str
nominee           str
artist            str
workers           str
img               str
winner           bool
dtype: object

## 2. Completeness

In [4]:
print("=== Spotify null counts ===")
print(spotify.isnull().sum()[spotify.isnull().sum() > 0])


=== Spotify null counts ===
artists       1
album_name    1
track_name    1
dtype: int64


In [5]:
print("=== Grammy null counts ===")
print(grammy.isnull().sum())
print()
print("artist null rate:", round(grammy['artist'].isnull().mean() * 100, 1), "%")
print("workers null rate:", round(grammy['workers'].isnull().mean() * 100, 1), "%")


=== Grammy null counts ===
year               0
title              0
published_at       0
updated_at         0
category           0
nominee            6
artist          1840
workers         2190
img             1367
winner             0
dtype: int64

artist null rate: 38.3 %
workers null rate: 45.5 %


**Finding:** Spotify is essentially complete (3 nulls total, same row). Grammy has
material nullness in `artist` (38%) and `workers` (46%) — expected for categories
that recognize a work rather than a named performer (e.g. liner notes, classical
ensembles), but it limits how many Grammy rows can carry a usable artist name for
matching against Spotify.

## 3. Uniqueness and Duplication

In [6]:
dup_full = spotify.duplicated().sum()
dup_track_id = spotify['track_id'].duplicated().sum()
print(f"Exact duplicate rows: {dup_full}")
print(f"Duplicate track_id (same song, different genre tag): {dup_track_id}")

example_id = spotify['track_id'].value_counts().index[0]
print()
print("Example — same track_id appearing under multiple genres:")
spotify[spotify['track_id'] == example_id][['track_id', 'track_name', 'artists', 'track_genre']]


Exact duplicate rows: 0
Duplicate track_id (same song, different genre tag): 24259

Example — same track_id appearing under multiple genres:


,track_id,track_name,artists,track_genre
8315,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,blues
19759,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,country
34728,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,folk
62226,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,j-pop
63087,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,j-rock
82064,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,power-pop
84129,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,psych-rock
99727,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,singer-songwriter
102732,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,songwriter


**Finding:** 0 exact duplicate rows, but 24,259 `track_id` duplicates. This is
expected: the same song is catalogued under several genres. The grain of the
Spotify fact table must therefore be (track_id, genre), not track_id alone —
confirmed by the example above.

In [7]:
dup_grammy = grammy.duplicated().sum()
print(f"Grammy exact duplicate rows: {dup_grammy}")


Grammy exact duplicate rows: 0


## 4. Categorical Content

In [8]:
print("Unique artists (Spotify):", spotify['artists'].nunique())
print("Unique genres (Spotify):", spotify['track_genre'].nunique())
print("Unique categories (Grammy):", grammy['category'].nunique())


Unique artists (Spotify): 31437
Unique genres (Spotify): 114
Unique categories (Grammy): 638


In [9]:
# Category naming inconsistencies: same category, different casing/wording
cats = pd.Series(grammy['category'].unique())
lower = cats.str.lower().str.strip()
dupe_mask = lower.duplicated(keep=False)
dupes = cats[dupe_mask].sort_values()

grouped = {}
for c in dupes:
    key = c.lower().strip()
    grouped.setdefault(key, []).append(c)

print(f"Category labels that collapse into the same category once normalized: {len(grouped)}")
for v in grouped.values():
    print(" ", v)


Category labels that collapse into the same category once normalized: 7
  ['Best Classical Performance - Instrumental Soloist OR Soloists (With Orchestra)', 'Best Classical Performance - Instrumental Soloist Or Soloists (With Orchestra)']
  ['Best Instrumental Soloist Performance (Without Orchestra)', 'Best Instrumental Soloist Performance (without Orchestra)']
  ['Best Instrumental Soloist(s) Performance (With Orchestra)', 'Best Instrumental Soloist(s) Performance (with Orchestra)']
  ['Best Small Ensemble Performance (With Or Without Conductor)', 'Best Small Ensemble Performance (with or without Conductor)']
  ['Best Southern, Country, Or Bluegrass Gospel Album', 'Best Southern, Country, or Bluegrass Gospel Album']
  ['Best Spoken Word Album For Children', 'Best Spoken Word Album for Children']
  ['Remixer Of The Year, Non-Classical', 'Remixer Of The Year, Non-classical', 'Remixer of the Year, Non-Classical']


**Finding:** 7 groups of Grammy categories are the same category written with
different capitalization or wording (normalizing spacing as well removes 9 redundant labels, 638 → 629) (e.g. "Non-Classical" vs "Non-classical" vs
"Non-classical"). Left un-normalized, these would be counted as separate
categories in any aggregation for R3.

## 5. Numerical Content

In [10]:
spotify[["popularity", "duration_ms", "danceability", "energy", "key", "loudness",
          "mode", "speechiness", "acousticness", "instrumentalness", "liveness",
          "valence", "tempo", "time_signature"]].describe().T[["min", "max", "mean"]]


,min,max,mean
popularity,0.000,100.000,33.238535
duration_ms,0.000,5237295.000,228029.153114
danceability,0.000,0.985,0.566800
energy,0.000,1.000,0.641383
key,0.000,11.000,5.309140
loudness,-49.531,4.532,-8.258960
mode,0.000,1.000,0.637553
speechiness,0.000,0.965,0.084652
acousticness,0.000,0.996,0.314910
instrumentalness,0.000,1.000,0.156050


In [11]:
zero_duration = (spotify["duration_ms"] == 0).sum()
print(f"Rows with duration_ms == 0 (impossible value): {zero_duration}")

out_of_range_popularity = ((spotify["popularity"] < 0) | (spotify["popularity"] > 100)).sum()
print(f"Rows with popularity outside [0, 100]: {out_of_range_popularity}")


Rows with duration_ms == 0 (impossible value): 1
Rows with popularity outside [0, 100]: 0


**Finding:** audio features are correctly bounded (0-1 ranges, plausible tempo
and loudness), but there is 1 row with `duration_ms == 0` — a physically
impossible value, and a direct candidate for a Critical validity rule.

## 6. Temporal Content

In [12]:
years = sorted(grammy["year"].unique())
missing_years = [y for y in range(min(years), max(years) + 1) if y not in years]
print(f"Grammy year coverage: {min(years)} - {max(years)}")
print(f"Missing years in range: {missing_years}")


Grammy year coverage: 1958 - 2019
Missing years in range: []


## 7. The `winner` Data Quality Issue (central finding)

In [13]:
print("Unique values in 'winner':", grammy["winner"].unique())
print("All rows winner == True:", (grammy["winner"] == True).all())


Unique values in 'winner': [ True]
All rows winner == True: True


In [14]:
rows_per_cat_year = grammy.groupby(["year", "category"]).size()

early = grammy[grammy["year"] < 2011]
late = grammy[grammy["year"] >= 2011]

print("Before 2011 - max rows per category/year:", early.groupby(["year", "category"]).size().max())
print("2011 onward - max rows per category/year:", late.groupby(["year", "category"]).size().max())
print()
print("Before 2011 - total rows:", early.shape[0])
print("2011 onward - total rows:", late.shape[0])


Before 2011 - max rows per category/year: 2
2011 onward - max rows per category/year: 8

Before 2011 - total rows: 3709
2011 onward - total rows: 1101


In [15]:
example = grammy[(grammy["category"] == "Record Of The Year") & (grammy["year"] == 2019)]
example[["nominee", "artist", "winner"]]


,nominee,artist,winner
0,Bad Guy,Billie Eilish,True
1,"Hey, Ma",Bon Iver,True
2,7 rings,Ariana Grande,True
3,Hard Place,H.E.R.,True
4,Talk,Khalid,True
5,Old Town Road,Lil Nas X Featuring Billy Ray Cyrus,True
6,Truth Hurts,Lizzo,True
7,Sunflower,Post Malone & Swae Lee,True


**Finding (central data quality issue):** `winner` contains only the value
`True` across all 4,810 rows — `False` never appears. Before 2011, each
category/year has at most 1-2 rows (only the winner was recorded). From 2011
onward, up to 8 rows appear per category/year (the full nominee list), but the
`winner` flag was never updated to mark non-winning nominees as `False`.
**`winner=True` cannot be trusted to mean "won" for 2011 onward.** This is
documented as rule DQ8 (Warning severity) — it does not block R1-R3, which do
not depend on `winner`, but it is essential evidence for anyone using this
dataset.

## 8. Cross-Source Consistency (Spotify ↔ Grammy)

In [16]:
import re, unicodedata

def norm(x):
    if not isinstance(x, str):
        return None
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode()
    return x.lower().strip()

spotify["track_norm"] = spotify["track_name"].map(norm)
spotify["artists_norm"] = spotify["artists"].map(norm)
grammy["nominee_norm"] = grammy["nominee"].map(norm)
grammy["artist_norm"] = grammy["artist"].map(norm)
grammy["grammy_idx"] = grammy.index

# Artist-level overlap (per distinct Grammy row with an artist)
has_artist = grammy["artist_norm"].notna()
print("Grammy rows with a null artist (can never match on artist):",
      int((~has_artist).sum()), f"({(~has_artist).mean() * 100:.1f}%)")


Grammy rows with a null artist (can never match on artist): 1840 (38.3%)


In [17]:
# Track-level match on exact normalized (title, artist string).
# Null keys are EXCLUDED: otherwise NaN would match NaN and create false matches.
sp_keys = spotify.dropna(subset=["track_norm", "artists_norm"])[["track_id", "track_norm", "artists_norm"]]
gr_keys = grammy.dropna(subset=["nominee_norm", "artist_norm"])

joined = gr_keys.merge(sp_keys, left_on=["nominee_norm", "artist_norm"],
                       right_on=["track_norm", "artists_norm"], how="inner")

distinct_matched = joined["grammy_idx"].nunique()
print(f"Rows produced by a naive join on (title, artist): {len(joined)}")
print(f"Distinct Grammy nominations matched:               {distinct_matched}"
      f" ({distinct_matched / len(grammy) * 100:.1f}% of {len(grammy)})")
print(f"Rows created by join fan-out (same song, several genre rows): {len(joined) - distinct_matched}")


Rows produced by a naive join on (title, artist): 443
Distinct Grammy nominations matched:               181 (3.8% of 4810)
Rows created by join fan-out (same song, several genre rows): 262


In [18]:
# Category label normalization (case + spacing)
key = grammy["category"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
print("Distinct raw category labels:       ", grammy["category"].nunique())
print("Distinct categories once normalized:", key.nunique())


Distinct raw category labels:        638
Distinct categories once normalized: 629


**Findings:**

- **The match rate must be counted per distinct Grammy nomination, not per joined row.**
  A naive join multiplies rows because Spotify repeats a song once per genre tag; counting
  joined rows overstates the match rate. Counted correctly, only about 4% of Grammy
  nominations match a Spotify track exactly (about 5.5% once titles and artists are
  normalized further; see `docs/transformation_integration.md`).
- **38% of Grammy rows have a null `artist` and can never match** by design.
- Most Grammy categories (album, producer, composer, classical work) are not individual
  songs and never match a `track_name`; unmatched nominations are kept, not dropped.
- **Integration consequence:** the join must be controlled so that one nomination maps to at
  most one Spotify track (candidate selection), otherwise `FactNomination` would be inflated.


## 9. Summary of Quality Risks Identified

| # | Dataset | Finding | Severity candidate |
|---|---|---|---|
| 1 | Spotify | `duration_ms == 0` (1 row) | Critical |
| 2 | Spotify | `track_id` duplicated across genres (24,259 rows) — expected, grain decision | Warning |
| 3 | Spotify | 3 null values in artists/album_name/track_name | Critical (if used as match key) |
| 4 | Grammy | `artist` null in 38% of rows | Warning |
| 5 | Grammy | `workers` null in 46% of rows | Informational |
| 6 | Grammy | 7 category labels inconsistent in casing/wording | Warning |
| 7 | Grammy | `winner` always `True`, unreliable from 2011 onward | Warning |
| 8 | Cross-source | About 4% exact / 5.5% normalized match rate per distinct nomination; 38% of Grammy rows have null artist; naive join multiplies rows | Critical for the join (fan-out), Informational for the rate |

These risks are formalized into explicit quality rules (with metric, threshold
and severity) in `docs/quality_rules.md`, and implemented as Great Expectations
Expectations in `gx/`.
